# Checkpoint A1: Chuẩn bị môi trường & Đặc tả bộ kiểm thử

Jupyter Notebook này giúp bạn chuẩn bị môi trường, kiểm tra kết nối các API (Gemini Cloud API & Ollama Local LLM) và định nghĩa 10 ca kiểm thử bảo mật dữ liệu PII.

---

### Bước 1: Tự động cài đặt thư viện & Cấu hình Windows Unicode
Chạy cell dưới đây để tự động kiểm tra và cài đặt các thư viện cần thiết, đồng thời cấu hình Unicode cho Windows console để tránh lỗi hiển thị tiếng Việt.

In [1]:
import sys
import subprocess

def install_package(package):
    try:
        # Trích xuất tên module gốc để kiểm tra import
        module_name = package.split('==')[0].split('>=')[0].replace('-', '_')
        if module_name == "google_generativeai":
            import google.generativeai
        elif module_name == "python_dotenv":
            import dotenv
        else:
            __import__(module_name)
    except ImportError:
        print(f"Chưa có thư viện '{package}'. Tiến hành cài đặt tự động...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"Đã cài đặt '{package}' thành công.")

# 1. Tự động cấu hình UTF-8 trên Windows console
if sys.platform.startswith("win"):
    try:
        sys.stdout.reconfigure(encoding="utf-8")
        sys.stderr.reconfigure(encoding="utf-8")
        print("[Windows] Đã cấu hình đầu ra stdout/stderr sang UTF-8 thành công.")
    except Exception as e:
        print(f"[Windows Warning] Không thể cấu hình UTF-8 cho console: {e}")

# 2. Cài đặt các thư viện cần thiết
required_packages = ["google-generativeai", "python-dotenv", "pytest", "requests"]
for pkg in required_packages:
    install_package(pkg)

print("\n=> [OK] Toàn bộ thư viện cơ bản đã sẵn sàng!")

Chưa có thư viện 'google-generativeai'. Tiến hành cài đặt tự động...
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
  Using cached googleapis_common_protos-1.75.0-py3-none-any.whl.metadata (8.6 kB)
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 11.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 25.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 18.5 MB/s  0:00:00m0:00:010:01
  Attempting uninstall: protobuf
    Found existing i

### Bước 2: Tự động khởi tạo tệp cấu hình môi trường `.env`
Chạy cell này để kiểm tra sự hiện diện của tệp `.env`. Nếu chưa có, tệp sẽ được tạo tự động với các cấu hình mẫu.

In [2]:
import os
from pathlib import Path

# Xác định thư mục session-06 chứa bài làm
current_dir = Path.cwd()
session_dir = None
for parent in [current_dir] + list(current_dir.parents):
    if parent.name == "session-06-compliance-capstone":
        session_dir = parent
        break

if not session_dir:
    session_dir = current_dir

dotenv_path = session_dir / ".env"

if not dotenv_path.exists():
    print(f"Tệp .env chưa tồn tại tại {dotenv_path}. Đang tự động khởi tạo...")
    dotenv_content = (
        "# Cấu hình khóa Gemini API và Ollama local\n"
        "GEMINI_API_KEY=YOUR_GEMINI_API_KEY_HERE\n"
        "OLLAMA_ENDPOINT=http://localhost:11434/v1/chat/completions\n"
    )
    dotenv_path.write_text(dotenv_content, encoding="utf-8")
    print(f"[LƯU Ý] Đã tạo file .env tại: {dotenv_path.name}\n" 
          "Hãy mở file này ra và điền GEMINI_API_KEY thực của bạn (nếu dùng Cloud API).")
else:
    print(f"[OK] Đã tìm thấy tệp tin .env tại: {dotenv_path.name}")

Tệp .env chưa tồn tại tại /Users/shimazu/Documents/9. active/course_viettel_network/vtn-ai-builders-bootcamp-2026/03-practice/day-03/session-06-compliance-capstone/.env. Đang tự động khởi tạo...
[LƯU Ý] Đã tạo file .env tại: .env
Hãy mở file này ra và điền GEMINI_API_KEY thực của bạn (nếu dùng Cloud API).


### Bước 3: Đặc tả bộ 10 ca kiểm thử bảo mật thông tin nhạy cảm (PII)
Định nghĩa 10 ca kiểm thử bao phủ đủ 4 nhóm tình huống: **bình thường**, **lỗi**, **thiếu dữ liệu**, **vượt phạm vi**.

In [ ]:
test_cases = {
    "TC-01": {
        "group": "binh_thuong",
        "mo_ta": "Email cá nhân trong câu bình thường",
        "dau_vao": "Vui lòng gửi hợp đồng đến nguyenvana@gmail.com",
        "pii_can_tim": "nguyenvana@gmail.com",
        "ket_qua_mong_doi": "[REDACTED_EMAIL]",
    },
    "TC-02": {
        "group": "binh_thuong",
        "mo_ta": "Số điện thoại di động Việt Nam",
        "dau_vao": "Liên hệ tôi qua số 0912345678 nhé",
        "pii_can_tim": "0912345678",
        "ket_qua_mong_doi": "[REDACTED_PHONE]",
    },
    "TC-03": {
        "group": "binh_thuong",
        "mo_ta": "Số CCCD trong văn bản",
        "dau_vao": "Căn cước công dân số 079201001234",
        "pii_can_tim": "079201001234",
        "ket_qua_mong_doi": "[REDACTED_CCCD]",
    },
    "TC-04": {
        "group": "loi",
        "mo_ta": "Email sai định dạng (thiếu @)",
        "dau_vao": "Gửi cho nguyenvana.gmail.com giúp tôi",
        "pii_can_tim": "(không ẩn)",
        "ket_qua_mong_doi": "Giữ nguyên câu đầu vào",
    },
    "TC-05": {
        "group": "loi",
        "mo_ta": "Số điện thoại quá dài",
        "dau_vao": "Gọi cho tôi số 0912345678910",
        "pii_can_tim": "(không ẩn)",
        "ket_qua_mong_doi": "Giữ nguyên câu đầu vào",
    },
    "TC-06": {
        "group": "loi",
        "mo_ta": "CCCD sai format chứa chữ cái",
        "dau_vao": "CCCD số 079abc01234xyz",
        "pii_can_tim": "(không ẩn)",
        "ket_qua_mong_doi": "Giữ nguyên câu đầu vào",
    },
    "TC-07": {
        "group": "thieu_du_lieu",
        "mo_ta": "Chuỗi đầu vào rỗng",
        "dau_vao": "",
        "pii_can_tim": "(không có)",
        "ket_qua_mong_doi": "Trả về chuỗi rỗng",
    },
    "TC-08": {
        "group": "thieu_du_lieu",
        "mo_ta": "Văn bản thường không chứa PII",
        "dau_vao": "Hôm nay thời tiết đẹp quá",
        "pii_can_tim": "(không có)",
        "ket_qua_mong_doi": "Giữ nguyên văn bản",
    },
    "TC-09": {
        "group": "vuot_pham_vi",
        "mo_ta": "Nhiều loại PII trong cùng câu",
        "dau_vao": "KH Trần Văn B, email tranb@yahoo.com, SĐT 0987654321, CCCD 079201005678",
        "pii_can_tim": "Trần Văn B, tranb@yahoo.com, 0987654321, 079201005678",
        "ket_qua_mong_doi": "Ẩn tất cả các trường PII tương ứng",
    },
    "TC-10": {
        "group": "vuot_pham_vi",
        "mo_ta": "Danh từ chung kỹ thuật trùng với PII",
        "dau_vao": "Email là viết tắt của Electronic Mail",
        "pii_can_tim": "(không ẩn)",
        "ket_qua_mong_doi": "Không ẩn nhầm từ 'Email'",
    },
}

print(f"Đã khởi tạo thành công {len(test_cases)} ca kiểm thử.")

In [ ]:
# Hiển thị danh sách ca kiểm thử và kiểm định độ bao phủ
from collections import Counter

group_labels = {
    "binh_thuong": "Bình thường",
    "loi": "Lỗi",
    "thieu_du_lieu": "Thiếu dữ liệu",
    "vuot_pham_vi": "Vượt phạm vi",
}

print(f"{'ID':<6} | {'Nhóm tình huống':<16} | {'Mô tả kiểm thử':<40} | {'PII cần tìm'}")
print("-" * 95)
for tid, tc in test_cases.items():
    lbl = group_labels.get(tc["group"], tc["group"])
    print(f"{tid:<6} | {lbl:<16} | {tc['mo_ta']:<40} | {tc['pii_can_tim']}")

# Assert kiểm định độ bao phủ
groups_found = Counter(tc["group"] for tc in test_cases.values())
required_groups = {"binh_thuong", "loi", "thieu_du_lieu", "vuot_pham_vi"}

assert len(test_cases) == 10, f"Phải định nghĩa đúng 10 ca kiểm thử! Hiện có {len(test_cases)}."
assert required_groups.issubset(set(groups_found.keys())), "Chưa bao phủ đủ 4 nhóm tình huống!"

print("\n=> [PASS] Bộ kiểm thử đạt tiêu chuẩn kiểm thử tối thiểu của Viettel Net.")

---
**Môi trường đã sẵn sàng!** Hãy tiếp tục với **[checkpoint-step-a2.ipynb](checkpoint-step-a2.ipynb)** để tiến hành chạy kiểm thử pass đầu tiên.